# Severity Prediction

## Purpose
Predict whether a COVID-19 VAERS adverse event report will be classified as **serious** (`SERIOUS = 1`) using structured clinical features and free-text symptom narratives.

## Models compared
- Logistic Regression (with TF-IDF text + structured features)
- Decision Tree
- Random Forest

## Data flow
1. Load pre-imputed VAERS dataset 
2. Load precomputed comorbidity indicators
3. Clean symptom text — separate pipelines for supervised (leakage-scrubbed) and clustering tasks
4. Build feature matrix: numeric + categorical + manufacturer + comorbidity + TF-IDF text
5. Tune models with stratified k-fold cross-validation and grid search
6. Compare structured-only vs structured+text feature sets
7. Extract and save feature importance for Logistic Regression, Random Forest, and Decision Tree


## 1. Setup & Imports

In [1]:
# 1. SETUP

import os
import re
import sys
import json
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import scipy.sparse as sp

warnings.filterwarnings("ignore")

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)

Python executable: /Users/ariellerothman/Desktop/Sperm Cell Analysis/.venv/bin/python
Python version: 3.12.10
Pandas version: 2.2.3


## 2. Load Data

Load the preprocessed VAERS dataset produced by `Data preparation.ipynb` (expected at `Outputs/df_clean_imputed.csv` or `Outputs/df_clean_engineered.csv`). If not found, re-run that notebook first.

In [ ]:
# 2. LOAD PREPROCESSED DATASET
from pathlib import Path

HERE = Path().resolve()  # project root (the folder containing this notebook)
candidate_paths = [
    str(HERE / "Outputs" / "df_clean_imputed.csv"),
    str(HERE / "Outputs" / "df_clean_engineered.csv"),
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_path is not None:
    df_clean = pd.read_csv(data_path, low_memory=False)
    print(f"✓ Loaded dataset from: {data_path}")
    print(f"  Shape: {df_clean.shape}")
else:
    raise FileNotFoundError(
        "Expected one of these files but none were found:\n"
        + "\n".join(candidate_paths)
    )

✓ Loaded dataset from: /Users/ariellerothman/Desktop/Capstone/Outputs/df_clean_imputed.csv
  Shape: (986096, 113)


## 3. Load Precomputed Comorbidity Indicators

In [ ]:
# ============================================================
# 3. LOAD PRECOMPUTED COMORBIDITY INDICATORS
# ============================================================

import os
import pandas as pd

from pathlib import Path
HERE   = Path().resolve()  # project root
OUTDIR = str(HERE / "Outputs")
os.makedirs(OUTDIR, exist_ok=True)

COMORB_PATH = str(HERE / "Outputs" / "comorbidity_indicators.csv")

# Detect currently available indicator columns in df_clean
existing_indicator_cols = [
    c for c in df_clean.columns
    if "__" in c or c.endswith("_MISSING")
]

print(f"Existing indicator columns already in df_clean: {len(existing_indicator_cols)}")

if not os.path.exists(COMORB_PATH):
    raise FileNotFoundError(
        f"Comorbidity indicator file not found: {COMORB_PATH}\n"
        "Run Data preparation notebook first to generate comorbidity_indicators.csv."
    )

comorb_df = pd.read_csv(COMORB_PATH, low_memory=False)

if "VAERS_ID" not in comorb_df.columns:
    raise ValueError("comorbidity_indicators.csv must contain VAERS_ID for merging.")

indicator_cols = [
    c for c in comorb_df.columns
    if c != "VAERS_ID" and ("__" in c or c.endswith("_MISSING"))
]

missing_in_df = [c for c in indicator_cols if c not in df_clean.columns]

if not missing_in_df:
    print("All comorbidity indicator columns are already present. Skipping merge.")
else:
    merge_cols = ["VAERS_ID"] + missing_in_df

    # Use one row per VAERS_ID to avoid accidental row multiplication
    comorb_merge = comorb_df[merge_cols].drop_duplicates(subset=["VAERS_ID"])

    df_clean = df_clean.merge(
        comorb_merge,
        on="VAERS_ID",
        how="left",
        validate="m:1"
    )

    # Fill unmatched IDs as 0 and cast to compact integer dtype
    for c in missing_in_df:
        df_clean[c] = df_clean[c].fillna(0).astype("int8")

    print(f"Merged {len(missing_in_df)} indicator columns from comorbidity_indicators.csv")

final_indicator_cols = [
    c for c in df_clean.columns
    if "__" in c or c.endswith("_MISSING")
]

print(f"Total indicator columns now in df_clean: {len(final_indicator_cols)}")

print("Sample indicator columns:")print(final_indicator_cols[:20])

Existing indicator columns already in df_clean: 64
All comorbidity indicator columns are already present. Skipping merge.
Total indicator columns now in df_clean: 64
Sample indicator columns:
['MANU__JANSSEN', 'MANU__MODERNA', 'MANU__NOVAVAX', 'MANU__PFIZER_BIONTECH', 'MANU__UNKNOWN_MANUFACTURER', 'AGE_YRS_MISSING', 'HOSPDAYS_MISSING', 'NUMDAYS_MISSING', 'ONSET_INTERVAL_MISSING', 'MAX_DOSE_MISSING', 'HISTORY_MISSING', 'HISTORY__cardiovascular', 'HISTORY__respiratory', 'HISTORY__metabolic', 'HISTORY__endocrine_thyroid', 'HISTORY__musculoskeletal', 'HISTORY__mental_health', 'HISTORY__gastrointestinal', 'HISTORY__kidney', 'HISTORY__clotting']


## 4. Symptom Text Preprocessing


| Column | Purpose |
|---|---|
| `SYMPTOM_TEXT_CLEAN` | Base cleaning (lowercased, URLs/numbers removed) |
| `SYMPTOM_TEXT_CLEAN_NOLEAK` | Supervised modelling — leakage terms scrubbed (hospitalization, death, outcome labels) |

In [4]:
# ============================================================
# 4. SYMPTOM TEXT PREPROCESSING
# ============================================================
# Produces two text columns for downstream use:
# - SYMPTOM_TEXT_CLEAN: base cleaning only
# - SYMPTOM_TEXT_CLEAN_NOLEAK: leakage-scrubbed for supervised modelling

import re
import pandas as pd

# --------------------------------------------------
# Base text cleaning
# --------------------------------------------------
_url_re = re.compile(r"http\S+|www\S+|https\S+")
_email_re = re.compile(r"\S+@\S+")
_num_re = re.compile(r"\d+")
_keep_re = re.compile(r"[^a-z0-9\s\-]")
_ws2 = re.compile(r"\s+")

def clean_symptom_text(x) -> str:
    """
    Base-level cleaning for symptom text before any model-specific scrubbing.

    Steps:
        1. Lowercase and strip whitespace
        2. Remove URLs and email addresses
        3. Replace numeric tokens with placeholder '__NUM__'
        4. Remove non-alphanumeric characters (except hyphens)
        5. Collapse multiple whitespace

    Args:
        x: Raw symptom text string (or NaN).

    Returns:
        Cleaned lowercase string, or empty string if input is null.
    """
    if pd.isna(x) or x == "":
        return ""
    x = str(x).strip().lower()
    x = _url_re.sub(" ", x)
    x = _email_re.sub(" ", x)
    x = _num_re.sub(" __NUM__ ", x)
    x = _keep_re.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN"] = df_clean["SYMPTOM_TEXT"].apply(clean_symptom_text)

# --------------------------------------------------
# Strong supervised leakage scrubbing for severity prediction
# Removes direct outcome / adjudication / downstream-care terms
# plus residual administrative/report-style phrases.
#
# WHY: The VAERS SERIOUS label is defined by hospitalization, death,
# disability, and life-threatening events. If those words appear
# verbatim in the symptom text, any classifier trivially learns to
# predict the label from them rather than real clinical patterns.
# --------------------------------------------------
LEAKAGE_PATTERNS = [re.compile(p) for p in [

    # ----------------------------------------------
    # Hospital / admission / downstream care
    # ----------------------------------------------
    r"\bhospital\w*\b",
    r"\badmit\w*\b",
    r"\binpatient\b",
    r"\bdischarg\w*\b",
    r"\bed\b",
    r"\ber\b",
    r"\bemergency\b",
    r"\bemergency room\b",
    r"\burgent\b",
    r"\burgent care\b",
    r"\btransport\w*\b",
    r"\btransfer\w*\b",
    r"\bambulance\b",
    r"\bems\b",
    r"\bhospice\b",

    # ----------------------------------------------
    # Death / life-threatening / rescue care
    # ----------------------------------------------
    r"\bdeath\b",
    r"\bdied\b",
    r"\bdead\b",
    r"\bfatal\b",
    r"\bpassed away\b",
    r"\bautopsy\b",
    r"\blife[-\s]?threaten\w*\b",
    r"\bintubat\w*\b",
    r"\bventilat\w*\b",
    r"\bicu\b",
    r"\barrest\b",

    # ----------------------------------------------
    # Seriousness / adjudication labels
    # ----------------------------------------------
    r"\bserious\w*\b",
    r"\bnon[-\s]?serious\w*\b",
    r"\bmedically significant\b",
    r"\bmedically important\b",
    r"\bcriterion\b",
    r"\bcriteria\b",
    r"\boutcome\b",
    r"\bdisability\b",
    r"\bpermanent\b",
    r"\bprolonged\b",

    # ----------------------------------------------
    # Administrative / report-template phrasing
    # ----------------------------------------------
    r"\breported cause\b",
    r"\breport medically\b",
    r"\bcriterion hospitalization\b",
    r"\bhospitalization medically\b",
    r"\bsignificant outcome\b",
    r"\bnarrative\b",
    r"\breport\b",
    r"\breported\b",
    r"\bspontaneous\b",
    r"\bcase\b",
    r"\bsource\b",
    r"\bpatient unknown\b",
    r"\bunknown\b",

    # ----------------------------------------------
    # Residual report-style / quasi-administrative phrases
    # seen in current model outputs
    # ----------------------------------------------
    r"\bevent resulted\b",
    r"\bevents resulted\b",
    r"\bresulted\b",
    r"\bcaused prolonged\b",
    r"\bmedical center\b",
    r"\bclinic care\b",
    r"\bwent care\b",
    r"\bvisited\b",
    r"\bperformed\b",

    # ----------------------------------------------
    # Other report-level phrasing seen previously
    # ----------------------------------------------
    r"\broom\b",
    r"\bdepartment\b",
    r"\bvisit\b",
    r"\bwent room\b",
    r"\bwent doctor\b",
    r"\bdoctor visit\b"
]]

def scrub_leakage_terms_strong(x) -> str:
    """
    Remove supervised leakage terms from cleaned symptom text.

    Args:
        x: Cleaned symptom text string (output of clean_symptom_text).

    Returns:
        Text with leakage terms replaced by spaces and whitespace collapsed.
    """
    if pd.isna(x) or x == "":
        return ""
    x = str(x)
    for p in LEAKAGE_PATTERNS:
        x = p.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN_NOLEAK"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(scrub_leakage_terms_strong)
)

print("Text columns created:")
print([c for c in df_clean.columns if "SYMPTOM" in c])


Text columns created:
['SYMPTOM_TEXT', 'SYMPTOM_TEXT_CHAR_LEN', 'SYMPTOM_TEXT_CLEAN', 'SYMPTOM_TEXT_CLEAN_NOLEAK']


## 5. Supervised Feature Matrix

Define which columns from `df_clean` enter the model as features. 


In [9]:
# 5. SUPERVISED FEATURE MATRIX SETUP
#     Cleaned final version for severity prediction

TEXT_FEATURE = "SYMPTOM_TEXT_CLEAN_NOLEAK"
y = df_clean["SERIOUS"].astype(int)

# Numeric features
# Keep onset-timing variables for the main model
# Remove HOSPDAYS because it is downstream / leakage-prone
numeric_cols = [
    c for c in ["AGE_YRS", "NUMDAYS", "ONSET_INTERVAL", "MAX_DOSE"]
    if c in df_clean.columns
]

# Coerce selected numeric columns to numeric dtype.
# This handles string artifacts like "UNKNOWN" from upstream exports.
for c in numeric_cols:
    df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

# Categorical features
categorical_cols = [
    c for c in ["SEX", "STATE", "YEAR", "MONTH"]
    if c in df_clean.columns
]

# Ensure categoricals are treated consistently as strings for the categorical imputer/encoder
for c in categorical_cols:
    df_clean[c] = df_clean[c].astype("string")

# Manufacturer and dose-related features
manufacturer_cols = [c for c in df_clean.columns if c.startswith("MANU__")]

dose_cols = [
    c for c in ["DOSE_COUNT", "MULTI_DOSE", "UNKNOWN_DOSE", "MULTI_MANUFACTURER"]
    if c in df_clean.columns
]

# Missingness indicators
missing_cols = [
    c for c in [
        "NUMDAYS_MISSING",
        "ONSET_INTERVAL_MISSING",
        "AGE_YRS_MISSING",
        "HISTORY_MISSING",
        "CUR_ILL_MISSING",
        "ALLERGIES_MISSING",
        "PRIOR_VAX_MISSING",
        "LAB_DATA_MISSING",
        "OTHER_MEDS_MISSING"
    ]
    if c in df_clean.columns
]

# Engineered clinical / comorbidity binary features
comorb_prefixes = (
    "HISTORY__",
    "CUR_ILL__",
    "ALLERGIES__",
    "PRIOR_VAX__",
    "LAB_DATA__",
    "OTHER_MEDS__"
)

comorb_cols = [c for c in df_clean.columns if c.startswith(comorb_prefixes)]

# Explicit exclusions for leakage-prone / helper / raw columns
exclude_cols = {
    # target / direct leakage
    "SERIOUS",
    "DIED",
    "L_THREAT",
    "ER_VISIT",
    "ER_ED_VISIT",
    "HOSPITAL",
    "DISABLE",
    "BIRTH_DEFECT",
    "HOSPDAYS",
    "DATEDIED",

    # identifiers / raw dates
    "VAERS_ID",
    "RECVDATE",
    "VAX_DATE",
    "ONSET_DATE",

    # raw free-text fields (already engineered elsewhere)
    "SYMPTOM_TEXT",
    "LAB_DATA",
    "OTHER_MEDS",
    "CUR_ILL",
    "HISTORY",
    "PRIOR_VAX",
    "ALLERGIES",

    # normalized helper text columns
    "HISTORY_NORM",
    "CUR_ILL_NORM",
    "ALLERGIES_NORM",
    "PRIOR_VAX_NORM",
    "LAB_DATA_NORM",
    "OTHER_MEDS_NORM",

    # text/clustering helper columns not used as structured inputs
    "SYMPTOM_TEXT_CLEAN",
    "SYMPTOM_TEXT_CLEAN_CLUSTER",
    "CLUSTER_TOKEN_COUNT",

    # optional: usually unhelpful if dataset is already one vaccine type
    "VAX_TYPE"
}

# Final structured feature list
structured_cols = (
    numeric_cols +
    categorical_cols +
    manufacturer_cols +
    dose_cols +
    missing_cols +
    comorb_cols
)

# Remove anything explicitly excluded
structured_cols = [c for c in structured_cols if c not in exclude_cols]

# Remove duplicates while preserving order
structured_cols = list(dict.fromkeys(structured_cols))

# Diagnostics
print("TEXT_FEATURE:", TEXT_FEATURE)
print("Target:", "SERIOUS")
print()

print("Numeric cols:", numeric_cols)
print("Categorical cols:", categorical_cols)
print("Manufacturer cols:", manufacturer_cols)
print("Dose cols:", dose_cols)
print("Missing cols:", missing_cols)
print("Comorbidity/clinical cols count:", len(comorb_cols))
print("Total structured feature count:", len(structured_cols))
print()

if numeric_cols:
    print("Numeric dtypes after coercion:")
    print(df_clean[numeric_cols].dtypes)

TEXT_FEATURE: SYMPTOM_TEXT_CLEAN_NOLEAK
Target: SERIOUS

Numeric cols: ['AGE_YRS', 'NUMDAYS', 'ONSET_INTERVAL', 'MAX_DOSE']
Categorical cols: ['SEX', 'STATE', 'YEAR', 'MONTH']
Manufacturer cols: ['MANU__JANSSEN', 'MANU__MODERNA', 'MANU__NOVAVAX', 'MANU__PFIZER_BIONTECH', 'MANU__UNKNOWN_MANUFACTURER']
Dose cols: ['DOSE_COUNT', 'MULTI_DOSE', 'UNKNOWN_DOSE', 'MULTI_MANUFACTURER']
Missing cols: ['NUMDAYS_MISSING', 'ONSET_INTERVAL_MISSING', 'AGE_YRS_MISSING', 'HISTORY_MISSING', 'CUR_ILL_MISSING', 'ALLERGIES_MISSING', 'PRIOR_VAX_MISSING', 'LAB_DATA_MISSING', 'OTHER_MEDS_MISSING']
Comorbidity/clinical cols count: 48
Total structured feature count: 74

Numeric dtypes after coercion:
AGE_YRS           float64
NUMDAYS           float64
ONSET_INTERVAL    float64
MAX_DOSE          float64
dtype: object


## 6. Column Inspection (Diagnostic)

Displays all columns in `df_clean` with their dtypes. Use this to verify that comorbidity indicators, text variants, and other engineered columns were created correctly before building the feature matrix.

In [10]:
# 6. COLUMN INSPECTION (DIAGNOSTIC)
import pandas as pd

print("Total columns:", len(df_clean.columns))

pd.set_option("display.max_rows", None)

col_df = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes
})

display(col_df)

Total columns: 115


,column,dtype
VAERS_ID,VAERS_ID,int64
DIED,DIED,int64
L_THREAT,L_THREAT,int64
ER_VISIT,ER_VISIT,int64
ER_ED_VISIT,ER_ED_VISIT,int64
HOSPITAL,HOSPITAL,int64
DISABLE,DISABLE,int64
BIRTH_DEFECT,BIRTH_DEFECT,int64
RECVDATE,RECVDATE,object
VAX_DATE,VAX_DATE,object


## 7. Preprocessors & Model Definitions

Build the scikit-learn preprocessing pipeline and define the three candidate classifiers.

**Models:**
- Logistic Regression 
- Decision Tree 
- Random Forest 

In [11]:
# ============================================================
# 7. PREPROCESSORS + MODELS
#    Raw TF-IDF version (NO SVD)
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# Structured preprocessing
# --------------------------------------------------
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

structured_binary_cols = [c for c in structured_cols if c not in numeric_cols + categorical_cols]

# Binary columns (comorbidity indicators, manufacturer flags, dose flags, missingness flags)
# are already 0/1 integers and need no transformation — passed through as-is.
# The step name "binary" appears as a prefix in feature importance output:
# e.g., binary__HISTORY__cardiovascular, binary__MANU__PFIZER/BIONTECH
structured_preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
        ("binary", "passthrough", structured_binary_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# --------------------------------------------------
# Raw TF-IDF preprocessing (NO SVD)
# --------------------------------------------------
text_preprocess = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,          # you can lower to 3000 if runtime is too heavy
        ngram_range=(1, 2),         # can switch to (1,1) if needed for speed
        min_df=10,
        max_df=0.9,
        stop_words="english",
        sublinear_tf=True,
        norm="l2",
        strip_accents="unicode",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b|__NUM__"
    ))
])

# --------------------------------------------------
# Combine structured + raw TF-IDF text
# --------------------------------------------------
full_preprocess = ColumnTransformer(
    transformers=[
        ("structured", structured_preprocess, structured_cols),
        ("text", text_preprocess, TEXT_FEATURE),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# Optional scaling for sparse matrix models
# MaxAbsScaler works well with sparse matrices
# Especially useful for logistic regression
raw_tfidf_pipeline_template = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression())   # placeholder; will be replaced per model
])

# --------------------------------------------------
# Models compatible with sparse raw TF-IDF
# --------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# --------------------------------------------------
# Parameter grids
# --------------------------------------------------
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1.0, 3.0]
    },
    "Decision Tree": {
        "clf__max_depth": [5, 10, 20],
        "clf__min_samples_split": [2, 5, 10]
    },
    "Random Forest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [2, 5]
    }
}

print("Models to run:", list(models.keys()))

Models to run: ['Logistic Regression', 'Decision Tree', 'Random Forest']


## 8. Model Training: Stratified K-Fold CV & Grid Search

Tune and evaluate each model using 3-fold stratified cross-validation on a stratified subset of up to 20,000 rows.

In [8]:
# 8. MODEL TRAINING: STRATIFIED K-FOLD CV + GRID SEARCH

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict, train_test_split
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Stratified subset for tuning
# --------------------------------------------------
subset_size = min(20000, len(df_clean))

df_tune, _ = train_test_split(
    df_clean,
    train_size=subset_size,
    stratify=df_clean["SERIOUS"],
    random_state=42
)

y_tune = df_tune["SERIOUS"].astype(int)

print("Tuning subset shape:", df_tune.shape)
print("Serious prevalence in tuning subset:", y_tune.mean())

# --------------------------------------------------
# 2. CV setup
# --------------------------------------------------
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = []
best_models = {}

# --------------------------------------------------
# 3. Lighter parameter grids
# --------------------------------------------------
light_param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1.0]
    },
    "Decision Tree": {
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [5, 10]
    },
    "Random Forest": {
        "clf__n_estimators": [100],
        "clf__max_depth": [10, 20]
    }
}

# --------------------------------------------------
# 4. Tune and evaluate
# --------------------------------------------------
for name, clf in models.items():
    print("\n" + "=" * 80)
    print("MODEL:", name)
    print("=" * 80)

    pipe = Pipeline([
        ("preprocess", full_preprocess),
        ("scale", MaxAbsScaler()),
        ("clf", clf)
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=light_param_grids[name],
        scoring="average_precision",
        cv=cv,
        n_jobs=1,
        verbose=1
    )

    grid.fit(df_tune, y_tune)
    best = grid.best_estimator_
    best_models[name] = best

    print("Best params:", grid.best_params_)

    y_pred = cross_val_predict(best, df_tune, y_tune, cv=cv, method="predict", n_jobs=1)
    y_prob = cross_val_predict(best, df_tune, y_tune, cv=cv, method="predict_proba", n_jobs=1)[:, 1]

    pr_auc = average_precision_score(y_tune, y_prob)
    f1 = f1_score(y_tune, y_pred, zero_division=0)
    precision = precision_score(y_tune, y_pred, zero_division=0)
    recall = recall_score(y_tune, y_pred, zero_division=0)
    cm = confusion_matrix(y_tune, y_pred)

    results.append({
        "Model": name,
        "PR-AUC": pr_auc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "BestParams": grid.best_params_,
        "ConfusionMatrix": cm
    })

    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print("Confusion Matrix:")
    print(cm)

summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "ConfusionMatrix"}
    for r in results
])

print("\nMODEL COMPARISON")
print(summary_df.sort_values("PR-AUC", ascending=False))

summary_df.to_csv(f"{OUTDIR}/supervised_model_comparison_raw_tfidf.csv", index=False)
print("\nStored best models:", list(best_models.keys()))


Tuning subset shape: (20000, 115)
Serious prevalence in tuning subset: 0.1985

MODEL: Logistic Regression
Fitting 3 folds for each of 2 candidates, totalling 6 fits


ValueError: 
All the 6 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/ariellerothman/Desktop/Sperm Cell Analysis/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/ariellerothman/Desktop/Sperm Cell Analysis/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ariellerothman/Desktop/Sperm Cell Analysis/.venv/lib/python3.12/site-packages/sklearn/pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
         ^^^^

## 9. Structured-Only vs Structured + Text

This tells us whether the text column meaningfully improves beyond what structured/clinical features alone capture.

In [ ]:
# ============================================================
# 9. STRUCTURED-ONLY VS STRUCTURED + RAW TF-IDF
# ============================================================

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

comparison_results = []

# --------------------------------------------------
# Structured-only pipeline
# --------------------------------------------------
structured_only_preprocess = ColumnTransformer(
    transformers=[
        ("structured", structured_preprocess, structured_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

structured_only_pipe = Pipeline([
    ("preprocess", structured_only_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

# --------------------------------------------------
# Structured + text pipeline
# --------------------------------------------------
structured_text_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

for label, pipe in {
    "Structured only": structured_only_pipe,
    "Structured + raw TF-IDF": structured_text_pipe
}.items():
    print("\nRunning:", label)

    y_pred = cross_val_predict(pipe, df_tune, y_tune, cv=cv, method="predict", n_jobs=1)
    y_prob = cross_val_predict(pipe, df_tune, y_tune, cv=cv, method="predict_proba", n_jobs=1)[:, 1]

    comparison_results.append({
        "FeatureSet": label,
        "PR-AUC": average_precision_score(y_tune, y_prob),
        "F1": f1_score(y_tune, y_pred, zero_division=0),
        "Precision": precision_score(y_tune, y_pred, zero_division=0),
        "Recall": recall_score(y_tune, y_pred, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison_results)
print("\nSTRUCTURED vs TEXT COMPARISON")
print(comparison_df.sort_values("PR-AUC", ascending=False))

comparison_df.to_csv(f"{OUTDIR}/structured_vs_raw_tfidf_comparison.csv", index=False)



Running: Structured only

Running: Structured + raw TF-IDF

STRUCTURED vs TEXT COMPARISON
                FeatureSet    PR-AUC        F1  Precision    Recall
1  Structured + raw TF-IDF  0.802686  0.703262   0.609245  0.831591
0          Structured only  0.541854  0.527446   0.430030  0.681922


## 10. Feature Importance

Extract and interpret the features driving each model's predictions:
- **Logistic Regression**: signed coefficients (positive = predicts serious; negative = predicts non-serious)
- **Random Forest**: Gini-based feature importances (unsigned, relative)
- **Decision Tree**: Gini importances + which features appear in actual tree splits + exported rule text (depth ≤ 4)

Results are saved separately for structured and text features to allow easy comparison of clinical vs. symptom-text signal.

In [ ]:
# 10A. FIT FINAL LOGISTIC REGRESSION MODEL FOR FEATURE IMPORTANCE

final_logreg_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

final_logreg_pipe.fit(df_tune, y_tune)

print("Final logistic regression model fitted on tuning subset.")


Final logistic regression model fitted on tuning subset.


In [ ]:
# 10B. EXTRACT TOP FEATURES FROM LOGISTIC REGRESSION
#     (structured + raw TF-IDF)

import numpy as np
import pandas as pd

# --------------------------------------------------
# Get fitted preprocessors
# --------------------------------------------------
preprocessor = final_logreg_pipe.named_steps["preprocess"]
clf = final_logreg_pipe.named_steps["clf"]

# Structured feature names
structured_feature_names = preprocessor.named_transformers_["structured"].get_feature_names_out()

# Text feature names
text_feature_names = preprocessor.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()

# Combine all feature names
all_feature_names = np.concatenate([structured_feature_names, text_feature_names])

# Logistic regression coefficients
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    "feature": all_feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "feature_type": ["structured"] * len(structured_feature_names) + ["text"] * len(text_feature_names)
})

# Top positive predictors of seriousness
top_positive = coef_df.sort_values("coefficient", ascending=False).head(50)

# Top negative predictors of seriousness
top_negative = coef_df.sort_values("coefficient", ascending=True).head(50)

# Top text-only predictors
top_text_positive = coef_df[coef_df["feature_type"] == "text"].sort_values("coefficient", ascending=False).head(50)
top_text_negative = coef_df[coef_df["feature_type"] == "text"].sort_values("coefficient", ascending=True).head(50)

print("\nTop positive predictors of SERIOUS=1")
print(top_positive[["feature", "coefficient", "feature_type"]])

print("\nTop negative predictors of SERIOUS=1")
print(top_negative[["feature", "coefficient", "feature_type"]])

print("\nTop POSITIVE text predictors")
print(top_text_positive[["feature", "coefficient"]])

print("\nTop NEGATIVE text predictors")
print(top_text_negative[["feature", "coefficient"]])

# Save results
coef_df.to_csv(f"{OUTDIR}/logreg_all_feature_coefficients_raw_tfidf.csv", index=False)
top_positive.to_csv(f"{OUTDIR}/logreg_top_positive_features_raw_tfidf.csv", index=False)
top_negative.to_csv(f"{OUTDIR}/logreg_top_negative_features_raw_tfidf.csv", index=False)
top_text_positive.to_csv(f"{OUTDIR}/logreg_top_positive_text_features_raw_tfidf.csv", index=False)
top_text_negative.to_csv(f"{OUTDIR}/logreg_top_negative_text_features_raw_tfidf.csv", index=False)



Top positive predictors of SERIOUS=1
               feature  coefficient feature_type
771               care     9.483316         text
5076              went     6.442492         text
873        clinic care     5.906450         text
2460                iv     5.611175         text
4449            stroke     5.450223         text
789              cause     5.412843         text
269          admission     5.354014         text
774       care patient     4.931352         text
1533               epi     4.421316         text
1535       epinephrine     4.361759         text
4150              sent     4.257962         text
3972          required     4.232653         text
3527         pneumonia     3.964075         text
766            cardiac     3.780355         text
1536            epipen     3.753000         text
478            arrived     3.680054         text
4810      unresponsive     3.665849         text
3447            pepcid     3.656146         text
3354      patient days     3.63

In [ ]:
# ============================================================
# 10C. RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

# Fit final random forest model
final_rf_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

final_rf_pipe.fit(df_tune, y_tune)

preprocessor_rf = final_rf_pipe.named_steps["preprocess"]
rf = final_rf_pipe.named_steps["clf"]

structured_feature_names_rf = preprocessor_rf.named_transformers_["structured"].get_feature_names_out()
text_feature_names_rf = preprocessor_rf.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()
all_feature_names_rf = np.concatenate([structured_feature_names_rf, text_feature_names_rf])

rf_importance_df = pd.DataFrame({
    "feature": all_feature_names_rf,
    "importance": rf.feature_importances_,
    "feature_type": ["structured"] * len(structured_feature_names_rf) + ["text"] * len(text_feature_names_rf)
}).sort_values("importance", ascending=False)

print("\nTop Random Forest features")
print(rf_importance_df.head(50))

rf_importance_df.to_csv(f"{OUTDIR}/random_forest_feature_importance_raw_tfidf.csv", index=False)



Top Random Forest features
                                  feature  importance feature_type
2                     num__ONSET_INTERVAL    0.051577   structured
102              binary__LAB_DATA_MISSING    0.046709   structured
1                            num__NUMDAYS    0.037669   structured
143   binary__LAB_DATA__vitals_procedures    0.018650   structured
0                            num__AGE_YRS    0.015372   structured
204                                 acute    0.014145         text
4246                                 site    0.013827         text
1059                                covid    0.011899         text
418                                   arm    0.011183         text
2376                            injection    0.010214         text
2385                       injection site    0.010021         text
144               binary__LAB_DATA__other    0.009525   structured
822                                 chest    0.009141         text
3574                            pr

In [ ]:
# 10D. FIT FINAL DECISION TREE MODEL

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MaxAbsScaler

# Fit final decision tree model
final_dt_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", DecisionTreeClassifier(
        max_depth=20,              # adjust based on your chosen best params
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ))
])

final_dt_pipe.fit(df_tune, y_tune)

print("Final decision tree model fitted.")


Final decision tree model fitted.


In [ ]:
# 10E. EXTRACT DECISION TREE FEATURE IMPORTANCE

# Get fitted objects
preprocessor_dt = final_dt_pipe.named_steps["preprocess"]
dt = final_dt_pipe.named_steps["clf"]

# Structured feature names
structured_feature_names_dt = preprocessor_dt.named_transformers_["structured"].get_feature_names_out()

# Text feature names
text_feature_names_dt = preprocessor_dt.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()

# Combine feature names
all_feature_names_dt = np.concatenate([structured_feature_names_dt, text_feature_names_dt])

# Importance values
dt_importances = dt.feature_importances_

# Put into dataframe
dt_importance_df = pd.DataFrame({
    "feature": all_feature_names_dt,
    "importance": dt_importances,
    "feature_type": ["structured"] * len(structured_feature_names_dt) + ["text"] * len(text_feature_names_dt)
})

# Sort descending
dt_importance_df = dt_importance_df.sort_values("importance", ascending=False)

# Top overall features
top_dt_features = dt_importance_df.head(50)

# Top text features only
top_dt_text_features = (
    dt_importance_df[dt_importance_df["feature_type"] == "text"]
    .sort_values("importance", ascending=False)
    .head(50)
)

# Top structured features only
top_dt_structured_features = (
    dt_importance_df[dt_importance_df["feature_type"] == "structured"]
    .sort_values("importance", ascending=False)
    .head(50)
)

print("\nTop overall Decision Tree features")
print(top_dt_features)

print("\nTop text Decision Tree features")
print(top_dt_text_features)

print("\nTop structured Decision Tree features")
print(top_dt_structured_features)

# Save results
dt_importance_df.to_csv(f"{OUTDIR}/decision_tree_feature_importance_raw_tfidf.csv", index=False)
top_dt_features.to_csv(f"{OUTDIR}/decision_tree_top_features_raw_tfidf.csv", index=False)
top_dt_text_features.to_csv(f"{OUTDIR}/decision_tree_top_text_features_raw_tfidf.csv", index=False)
top_dt_structured_features.to_csv(f"{OUTDIR}/decision_tree_top_structured_features_raw_tfidf.csv", index=False)



Top overall Decision Tree features
                                  feature  importance feature_type
1                            num__NUMDAYS    0.229935   structured
102              binary__LAB_DATA_MISSING    0.172037   structured
961                           concomitant    0.052219         text
749                                called    0.028635         text
144               binary__LAB_DATA__other    0.028158   structured
0                            num__AGE_YRS    0.022159   structured
771                                  care    0.019854         text
2460                                   iv    0.018481         text
822                                 chest    0.013591         text
789                                 cause    0.011609         text
1562                           evaluation    0.010547         text
204                                 acute    0.010265         text
418                                   arm    0.009250         text
1590                      

In [ ]:
# 10F. FEATURES USED IN DECISION TREE SPLITS

used_feature_idx = dt.tree_.feature
used_feature_idx = used_feature_idx[used_feature_idx >= 0]   # remove leaf markers (-2)

used_feature_names = all_feature_names_dt[used_feature_idx]
used_feature_names_unique = pd.Series(used_feature_names).value_counts().reset_index()
used_feature_names_unique.columns = ["feature", "num_splits"]

print("\nFeatures actually used in tree splits")
print(used_feature_names_unique.head(50))

used_feature_names_unique.to_csv(
    f"{OUTDIR}/decision_tree_features_used_in_splits_raw_tfidf.csv",
    index=False
)


Features actually used in tree splits
                     feature  num_splits
0               num__AGE_YRS          78
1        num__ONSET_INTERVAL          31
2                      covid          30
3                    patient          26
4                        arm          25
5                    vaccine          22
6                   received          20
7                       care          19
8                       went          18
9                       days          16
10                  headache          16
11              num__NUMDAYS          15
12                  symptoms          15
13                      rash          14
14                   started          14
15                    tested          14
16                      pain          14
17                      time          14
18                    chills          13
19                     weeks          13
20                      dose          13
21                    doctor          13
22                

In [ ]:
# 10G. EXPORT DECISION TREE RULES

from sklearn.tree import export_text

tree_rules = export_text(
    dt,
    feature_names=list(all_feature_names_dt),
    max_depth=4
)

print(tree_rules)

with open(f"{OUTDIR}/decision_tree_rules_raw_tfidf.txt", "w") as f:
    f.write(tree_rules)


|--- num__NUMDAYS <= 0.00
|   |--- binary__LAB_DATA_MISSING <= 0.50
|   |   |--- binary__LAB_DATA__other <= 0.50
|   |   |   |--- date delivery <= 0.07
|   |   |   |   |--- appropriate section <= 0.41
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- appropriate section >  0.41
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- date delivery >  0.07
|   |   |   |   |--- inhaler <= 0.15
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |   |--- inhaler >  0.15
|   |   |   |   |   |--- class: 1
|   |   |--- binary__LAB_DATA__other >  0.50
|   |   |   |--- concomitant <= 0.21
|   |   |   |   |--- iv <= 0.03
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- iv >  0.03
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |--- concomitant >  0.21
|   |   |   |   |--- care <= 0.03
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- care >  0.03
|   |   |   |   |   |--- trunc